# Interactive Decision Tree - Separate Train/Test Demo

Bu notebook ayri train ve ayri test datasini UI'da birlikte denemek icin hazirlandi.

Akis:

1. Notebook icinde train ve test DataFrame'leri ayri ayri uretilir.
2. Train DataFrame `launch_tree(...)` ile UI'a gonderilir.
3. Test DataFrame ayri bir `data_id` ile session snapshot olarak kaydedilir.
4. UI'da `Data source` panelinde `Test / validation source = Separate data source` secilip test `data_id` girilir.
5. Istersen ayni train/test setleri CSV/Excel veya lokal SQLite SQL uzerinden de denenebilir.
6. UI'dan export edilen final pickle/JSON notebook'a geri yuklenip tek musteri skorlanir.


## 0. Kurulum notu

Bu notebook'u repo kokunden calistiriyorsan once bir kez su kurulum yeterli olur:

```powershell
.\.venv\Scripts\python.exe -m pip install -e ".[notebook]"
```

VS Code/Jupyter kernel olarak `interactive_decision_tree_env (.venv)` secili olmali. Import hatasi alirsan asagidaki `%pip install` satirini acip bir kez calistir.


In [ ]:
# Import hatasi alirsan once kernel'in `interactive_decision_tree_env (.venv)` oldugunu kontrol et.
# Gerekirse bu satiri acip bir kez calistir:
# %pip install -e ".[notebook]"

import inspect
import sys
from pathlib import Path
from urllib.parse import parse_qs, urlsplit, urlunsplit

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "interactive_decision_tree").exists():
            PROJECT_ROOT = candidate
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from interactive_decision_tree import launch_tree, load_tree_json, load_tree_pickle, score_tree_payload
    from interactive_decision_tree.session_store import default_session_dir, save_dataframe_session
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Notebook kernel'inde eksik paket var veya yanlis kernel secili. "
        "VS Code'da kernel olarak `interactive_decision_tree_env (.venv)` sec. "
        "Gerekirse bu hucreden once `%pip install -e \".[notebook]\"` calistir."
    ) from exc

PROJECT_ROOT


## 0.1 UI link ayarlari

Lokal makinede default ayarlar yeterli. OpenShift/Jupyter proxy veya route kullaniyorsan `APP_BASE_URL`, `APP_HOST` ve `APP_SCHEME` alanlarini burada degistir.


In [ ]:
APP_PORT = 8501
APP_START_SERVER = True
APP_OPEN_BROWSER = True

# OpenShift/Jupyter proxy icin ornek:
# APP_START_SERVER = False
# APP_OPEN_BROWSER = False
# APP_BASE_URL = "https://<notebook-host>/notebook/<workspace>/proxy/8501/"
APP_BASE_URL = ""

# Route veya farkli host icin ornek:
# APP_HOST = "interactive-tree.apps.internal"
# APP_SCHEME = "https"
APP_HOST = "localhost"
APP_SCHEME = "http"


def ui_launch_kwargs(func=launch_tree) -> dict:
    accepted = set(inspect.signature(func).parameters)
    kwargs = {
        "port": APP_PORT,
        "open_browser": APP_OPEN_BROWSER,
        "start_server": APP_START_SERVER,
    }
    if "host" in accepted:
        kwargs["host"] = APP_HOST
    if "scheme" in accepted:
        kwargs["scheme"] = APP_SCHEME
    if APP_BASE_URL and "base_url" in accepted:
        kwargs["base_url"] = APP_BASE_URL
    return kwargs


def format_ui_url(url: str) -> str:
    if not APP_BASE_URL:
        return url
    query = urlsplit(url).query
    base = urlsplit(APP_BASE_URL)
    return urlunsplit((base.scheme, base.netloc, base.path or "/", query, base.fragment))


def data_id_from_url(url: str) -> str:
    parsed = parse_qs(urlsplit(url).query)
    values = parsed.get("data_id") or []
    if not values:
        raise ValueError(f"URL icinde data_id yok: {url}")
    return values[0]


ui_launch_kwargs()


## 1. Ayri train ve test DataFrame uretme

Train ve test ayni kolon sozlesmesine sahip, fakat farkli random seed ve hafif farkli dagilim ile uretilir. Boylece UI'da train/test performans metriklerini gercekten karsilastirabilirsin.


In [ ]:
TARGET = "risk_flag"
FEATURES = ["age", "income", "tenure_months", "segment", "channel", "region", "utilization"]


def make_customer_frame(n: int, seed: int, *, income_shift: float = 0.0, mobile_boost: float = 0.0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    age = rng.integers(21, 76, size=n)
    income = rng.normal(55_000 + income_shift, 19_000, size=n).clip(10_000, 160_000).round(2)
    tenure_months = rng.integers(0, 132, size=n)
    segment = rng.choice(["A", "B", "C", "D"], size=n, p=[0.22, 0.34, 0.28, 0.16])
    channel = rng.choice(
        ["branch", "mobile", "web", "call_center"],
        size=n,
        p=[0.24 - mobile_boost / 2, 0.34 + mobile_boost, 0.30 - mobile_boost / 2, 0.12],
    )
    region = rng.choice(["marmara", "ege", "ic_anadolu", "akdeniz", "karadeniz"], size=n)
    utilization = rng.beta(2.2, 4.0, size=n).round(4)

    logit = (
        -1.15
        + (income < 38_000) * 1.15
        + (tenure_months < 18) * 0.65
        + (segment == "C") * 0.75
        + (segment == "D") * 0.45
        + (channel == "mobile") * 0.35
        + (utilization > 0.62) * 1.10
        + rng.normal(0, 0.32, size=n)
    )
    probability = 1 / (1 + np.exp(-logit))
    risk_flag = np.where(rng.random(n) < probability, "high_risk", "low_risk")

    return pd.DataFrame(
        {
            "age": age,
            "income": income,
            "tenure_months": tenure_months,
            "segment": segment,
            "channel": channel,
            "region": region,
            "utilization": utilization,
            "risk_flag": risk_flag,
        }
    )


train_df = make_customer_frame(900, seed=2026)
test_df = make_customer_frame(300, seed=2027, income_shift=-2_500, mobile_boost=0.08)


def profile_dataset(name: str, df: pd.DataFrame) -> dict:
    return {
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "high_risk_rate": float((df[TARGET] == "high_risk").mean()),
        "income_mean": float(df["income"].mean()),
        "mobile_share": float((df["channel"] == "mobile").mean()),
    }


pd.DataFrame([profile_dataset("train", train_df), profile_dataset("test", test_df)])


## 2. Session DataFrame ile ayri train/test baglama

Bu hucre train datasini UI'a yollar, test datasini ise ayri `data_id` olarak kaydeder.

UI'da test datasini baglamak icin:

1. `Data source` panelini ac.
2. `Test / validation source` = `Separate data source` sec.
3. `Test data source` = `Session DataFrame` sec.
4. Asagida yazan `test_data_id` degerini `Session data_id` alanina gir.


In [ ]:
test_data_id, test_metadata = save_dataframe_session(
    test_df,
    source="notebook_test",
    name="Separate holdout test data",
    target=TARGET,
    features=FEATURES,
)

train_url = format_ui_url(
    launch_tree(
        train_df,
        target=TARGET,
        features=FEATURES,
        session_name="Separate train data",
        **ui_launch_kwargs(launch_tree),
    )
)
train_data_id = data_id_from_url(train_url)

print("UI URL:", train_url)
print("train_data_id:", train_data_id)
print("test_data_id:", test_data_id)
print("session_dir:", default_session_dir())
pd.DataFrame(
    [
        {"dataset": "train", "data_id": train_data_id, "rows": len(train_df)},
        {"dataset": "test", "data_id": test_data_id, "rows": len(test_df)},
    ]
)


## 3. CSV / Excel ile ayri train/test upload testi

UI'da ayni senaryoyu dosya upload ile test etmek istersen bu hucre dosyalari `examples/generated_train_test_data/` altina yazar.

UI kullanimi:

- Train icin `Train data source = CSV / Excel Upload` secip train dosyasini yukle.
- Test icin `Test / validation source = Separate data source`, `Test data source = CSV / Excel Upload` secip test dosyasini yukle.


In [ ]:
output_dir = PROJECT_ROOT / "examples" / "generated_train_test_data"
output_dir.mkdir(parents=True, exist_ok=True)

train_csv_path = output_dir / "separate_train_sample.csv"
test_csv_path = output_dir / "separate_test_sample.csv"
train_xlsx_path = output_dir / "separate_train_sample.xlsx"
test_xlsx_path = output_dir / "separate_test_sample.xlsx"

train_df.to_csv(train_csv_path, index=False)
test_df.to_csv(test_csv_path, index=False)
train_df.to_excel(train_xlsx_path, index=False)
test_df.to_excel(test_xlsx_path, index=False)

pd.DataFrame(
    [
        {"kind": "train_csv", "path": str(train_csv_path), "rows": len(train_df)},
        {"kind": "test_csv", "path": str(test_csv_path), "rows": len(test_df)},
        {"kind": "train_xlsx", "path": str(train_xlsx_path), "rows": len(train_df)},
        {"kind": "test_xlsx", "path": str(test_xlsx_path), "rows": len(test_df)},
    ]
)


## 4. SQL ile ayri train/test testi (lokal SQLite)

Kurumsal DB'ye gerek kalmadan UI'daki SQL train/test akisini test etmek icin ayni datalari lokal SQLite dosyasina yazar.

UI kullanimi:

- Train icin `Train data source = SQL`, connection URL olarak asagidaki `sqlite_url` degerini gir.
- SQL mode `Query` sec ve train query'yi kullan.
- Test icin `Test / validation source = Separate data source`, `Test data source = SQL` sec.
- Ayni connection URL ile test query'yi kullan.


In [ ]:
from sqlalchemy import create_engine, text

sqlite_path = output_dir / "separate_train_test_demo.sqlite"
sqlite_url = f"sqlite:///{sqlite_path.as_posix()}"

engine = create_engine(sqlite_url)
try:
    train_df.to_sql("idt_train_demo", con=engine, if_exists="replace", index=False)
    test_df.to_sql("idt_test_demo", con=engine, if_exists="replace", index=False)
    with engine.connect() as conn:
        train_rows = conn.execute(text("select count(*) from idt_train_demo")).scalar_one()
        test_rows = conn.execute(text("select count(*) from idt_test_demo")).scalar_one()
finally:
    engine.dispose()

train_query = "select * from idt_train_demo"
test_query = "select * from idt_test_demo"

print("sqlite_url:", sqlite_url)
print("train_query:", train_query)
print("test_query:", test_query)
print({"train_rows": int(train_rows), "test_rows": int(test_rows)})


## 5. UI'dan final agaci notebook'a geri yukleme

UI'da agaci finalize ettikten sonra en alttaki `Tree export` bolumunden `Download runnable tree pickle` veya JSON indir. Sonra dosyayi bu hucreyle notebook'a al.


In [ ]:
tree_pickle_path = Path.home() / "Downloads" / "interactive_entropy_tree_runnable.pkl"
tree_json_path = Path.home() / "Downloads" / "interactive_entropy_tree_runnable.json"

if tree_pickle_path.exists():
    tree_payload = load_tree_pickle(tree_pickle_path)
    loaded_tree_path = tree_pickle_path
elif tree_json_path.exists():
    tree_payload = load_tree_json(tree_json_path)
    loaded_tree_path = tree_json_path
else:
    raise FileNotFoundError(
        "Downloads klasorunde interactive_entropy_tree_runnable.pkl veya .json bulunamadi. "
        "Once UI'dan final agaci indir."
    )

print("loaded_tree_path:", loaded_tree_path)
print("target:", tree_payload.get("target"))
print("features:", tree_payload.get("features"))
print("node_count:", tree_payload.get("node_count"))
print("split_count:", tree_payload.get("split_count"))


## 6. Test datasindan tek musteri skorlama

Bu hucre final pickle/JSON ile test datasindaki bir musteri icin prediction ve probability uretir.


In [ ]:
if "tree_payload" not in globals():
    raise RuntimeError("Once UI export pickle/JSON dosyasini tree_payload degiskenine yukle.")

one_customer = test_df.drop(columns=[TARGET]).iloc[0].to_dict()
score_result = score_tree_payload(tree_payload, one_customer)

print("input_customer:")
display(pd.DataFrame([one_customer]))

print("prediction:", score_result["prediction"])
print("prediction_proba:", score_result["prediction_probability"])
print("positive_class:", score_result["positive_class"])
print("positive_class_proba:", score_result["positive_class_probability"])
print("leaf_node_id:", score_result["leaf_node_id"])
print("leaf_path:", score_result["leaf_path"])

display(pd.DataFrame([score_result["class_probabilities"]]))
pd.DataFrame(score_result["trace"])
